In [7]:
import cv2
import os
import glob
import numpy as np

# -----------------------------
# Update paths here
# -----------------------------
GOLDEN_IMG = r"C:\\Users\\surya\\OneDrive\\Desktop\\jupyter\\PCB_DATASET\\PCB_USED\\01.JPG"  # Defect-free PCB
DEFECT_ROOT = r"C:\\Users\\surya\\OneDrive\\Desktop\\jupyter\\PCB_DATASET\\images"         # Defect types folders
OUTPUT_ROOT = r"C:\\Users\\surya\\OneDrive\\Desktop\\jupyter\\PCB_DATASET\\Module1"                     # Output binary masks folder

# Create output root if missing
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# -----------------------------
# Load golden/reference image
# -----------------------------
golden = cv2.imread(GOLDEN_IMG)
if golden is None:
    raise FileNotFoundError(f"Golden image not found: {GOLDEN_IMG}")
golden_gray = cv2.cvtColor(golden, cv2.COLOR_BGR2GRAY)

print("Golden PCB loaded:", GOLDEN_IMG)

# -----------------------------
# Subtraction and mask extraction function
# -----------------------------
def process_image(defect_img_path, save_dir):
    img = cv2.imread(defect_img_path)
    if img is None:
        print(f"⚠️ Skipping {defect_img_path} (cannot load)")
        return
    
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Resize if size mismatch
    if img_gray.shape != golden_gray.shape:
        img_gray = cv2.resize(img_gray, (golden_gray.shape[1], golden_gray.shape[0]))
    
    # Image subtraction
    diff = cv2.absdiff(img_gray, golden_gray)
    
    # Otsu's thresholding
    _, mask = cv2.threshold(diff, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # Noise removal with morphological opening
    kernel = np.ones((3,3), np.uint8)
    mask_clean = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    
    # Save mask image
    base = os.path.splitext(os.path.basename(defect_img_path))[0]
    cv2.imwrite(os.path.join(save_dir, f"{base}_mask.png"), mask_clean)

# -----------------------------
# Process all defect images
# -----------------------------
defect_types = [d for d in os.listdir(DEFECT_ROOT) if os.path.isdir(os.path.join(DEFECT_ROOT, d))]

for defect in defect_types:
    defect_dir = os.path.join(DEFECT_ROOT, defect)
    save_dir = os.path.join(OUTPUT_ROOT, defect)
    os.makedirs(save_dir, exist_ok=True)

    imgs = sorted(glob.glob(os.path.join(defect_dir, "*.jpg")) + 
                  glob.glob(os.path.join(defect_dir, "*.png")))
    
    print(f"Processing {len(imgs)} images in {defect}...")
    
    for img_path in imgs:
        process_image(img_path, save_dir)

print("\n✅ Module 1 complete! Binary defect masks saved.")


Golden PCB loaded: C:\\Users\\surya\\OneDrive\\Desktop\\jupyter\\PCB_DATASET\\PCB_USED\\01.JPG
Processing 115 images in Missing_hole...
Processing 0 images in Module1_Samples...
Processing 115 images in Mouse_bite...
Processing 116 images in Open_circuit...
Processing 116 images in Short...
Processing 115 images in Spur...
Processing 116 images in Spurious_copper...

✅ Module 1 complete! Binary defect masks saved.
